# DeepGuard v2 · Official FaceForensics++ download → Google Drive

**What this notebook does**
1. Downloads the official FF++ + DeepFakeDetection videos (c23 and c40) with the authors' script, straight into `My Drive/DeepGuard/raw/ffpp/`. Nothing touches your laptop.
2. Writes a SHA-256 `MANIFEST.csv` for every file (our Layer-1 checksum record, D-013).
3. Compares those checksums with the Kaggle copy (from the audit notebook's `MANIFEST.csv`). If every file is byte-identical, Kaggle notebooks can keep using the Kaggle copy and we save ~45 GB of uploads.

**Before running**
- Colab: **Runtime → Change runtime type → CPU** (no GPU needed).
- Colab Secrets (key icon on the left): add **`FFPP_SCRIPT_URL`** = the `faceforensics_download_v4.py` link from the approval email, and switch on *Notebook access*. The link stays out of the notebook and out of GitHub.
- Drive needs ~45 GB free (you have 5 TB).
- Keep the tab open while it downloads (Colab disconnects idle tabs). If it disconnects, just **Run all** again: finished jobs are skipped.

In [ ]:
# ===== CELL 1: Setup — mount Google Drive, define folders =====
import os, sys, re, json, time, glob, hashlib, subprocess, zipfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

T0 = time.time()
def log(msg):
    """Print with elapsed time so progress is visible."""
    m, s = divmod(int(time.time() - T0), 60)
    print(f"[{m:3d}m{s:02d}s] {msg}", flush=True)

LOCAL_TEST = os.environ.get("DG_LOCAL_TEST") == "1"     # only for offline testing of this notebook
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = Path(os.environ.get("DG_DRIVE", "/content/drive/MyDrive"))
RAW = DRIVE / "DeepGuard" / "raw" / "ffpp"          # official copy lives here (never modified after download)
AUDIT = DRIVE / "DeepGuard" / "audit"               # put ffpp_audit_results.zip from Kaggle here
RAW.mkdir(parents=True, exist_ok=True); AUDIT.mkdir(parents=True, exist_ok=True)
STATUS = RAW / "_download_status.json"               # which download jobs have finished
SCRIPT = Path("/content/ffpp_download.py") if not LOCAL_TEST else Path(os.environ["DG_SCRIPT_PATH"])
log(f"Drive folder for the official copy: {RAW}")

In [ ]:
# ===== CELL 2: Get the official download script and show its options =====
# The link comes from the Colab secret FFPP_SCRIPT_URL (from the approval email).
if not LOCAL_TEST:
    from google.colab import userdata
    url = userdata.get("FFPP_SCRIPT_URL").strip()
    urllib.request.urlretrieve(url, SCRIPT)
log(f"Script saved: {SCRIPT} ({SCRIPT.stat().st_size:,} bytes)")
help_txt = subprocess.run([sys.executable, str(SCRIPT), "--help"], capture_output=True, text=True, input="\n").stdout
print(help_txt)
m = re.search(r"-d\s+\{([^}]*)\}", help_txt) or re.search(r"--dataset\s+\{([^}]*)\}", help_txt)
SCRIPT_DATASETS = m.group(1).split(",") if m else []
log(f"Datasets the script offers: {SCRIPT_DATASETS or 'could not read — check the help text above'}")

In [ ]:
# ===== CELL 3: What to download (in priority order) =====
# videos only (no raw, no masks for now). c23 first because it is what we train on.
SERVER = "EU2"            # the approval email says EU2 is the only working server
JOBS = [(d, c) for c in ["c23", "c40"] for d in
        ["original", "Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter",
         "DeepFakeDetection_original", "DeepFakeDetection"]]
# where each dataset lands inside RAW (official folder layout)
FOLDER = {"original": "original_sequences/youtube", "DeepFakeDetection_original": "original_sequences/actors",
          **{d: f"manipulated_sequences/{d}" for d in ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures",
                                                       "FaceShifter", "DeepFakeDetection"]}}
EXPECTED = {"original": 1000, "DeepFakeDetection_original": 363, "DeepFakeDetection": 3068,
            **{d: 1000 for d in ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]}}
missing = [d for d, _ in JOBS if SCRIPT_DATASETS and d not in SCRIPT_DATASETS]
assert not missing, f"The script does not offer {set(missing)} — send Claude the help text from Cell 2"
status = json.load(open(STATUS)) if STATUS.exists() else {}
for d, c in JOBS:
    print(f"  {d:28s} {c}  →  {FOLDER[d]}/{c}/videos   {'DONE' if status.get(f'{d}|{c}') else 'to do'}")

In [ ]:
# ===== CELL 4: Download — several parts in parallel, resumable, progress every 30 s =====
# The official script downloads one file at a time (into a temporary 'tmp…' file, then renames it to .mp4)
# and skips files that already exist, so re-running this cell continues where it stopped.
MAX_PARALLEL = 4                                   # parts downloading at the same time
POLL = int(os.environ.get("DG_POLL", "30"))        # seconds between progress prints

# 1) stop any downloader left running by an earlier (crashed) run of this cell, then remove its half-written temp files
subprocess.run(["pkill", "-f", str(SCRIPT)], capture_output=True)
time.sleep(2)
removed = 0
for d, c in JOBS:
    for f in (RAW / FOLDER[d] / c / "videos").glob("tmp*"):
        f.unlink(missing_ok=True); removed += 1
log(f"Stopped old downloaders; removed {removed} half-written temp files")

def mp4_stats(p):
    # count finished .mp4 files only; files can vanish while we look (temp → rename), so ignore those
    n, size = 0, 0
    for f in Path(p).glob("*.mp4") if Path(p).exists() else []:
        try:
            size += f.stat().st_size; n += 1
        except FileNotFoundError:
            pass
    return n, size / 1e9

def start(d, c):
    cmd = [sys.executable, str(SCRIPT), str(RAW), "-d", d, "-c", c, "-t", "videos", "--server", SERVER]
    logf = open(RAW / f"_log_{d}_{c}.txt", "a")
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=logf, stderr=subprocess.STDOUT, text=True)
    p.stdin.write("\n\n"); p.stdin.close()          # answers the "press any key to accept the terms" prompt
    log(f"START {d} {c}")
    return p, logf

queue = [(d, c) for d, c in JOBS if not status.get(f"{d}|{c}")]
running = {}
t_start = time.time(); gb_start = sum(mp4_stats(RAW / FOLDER[d] / c / "videos")[1] for d, c in JOBS)
while queue or running:
    while queue and len(running) < MAX_PARALLEL:          # fill free slots
        job = queue.pop(0); running[job] = start(*job)
    time.sleep(POLL)
    for (d, c), (p, logf) in list(running.items()):
        n, gb = mp4_stats(RAW / FOLDER[d] / c / "videos")
        if p.poll() is None:
            print(f"      {d:27s} {c}: {n:5d}/{EXPECTED[d]} files, {gb:6.2f} GB", flush=True)
            continue
        logf.close(); del running[(d, c)]
        ok = p.returncode == 0 and n >= EXPECTED[d]
        status[f"{d}|{c}"] = ok
        json.dump(status, open(STATUS, "w"), indent=1)
        log(f"END {d} {c}: exit {p.returncode}, {n}/{EXPECTED[d]} files, {gb:.2f} GB → {'OK' if ok else 'CHECK'}")
        if not ok:
            print("   last lines of its log:\n" + open(RAW / f"_log_{d}_{c}.txt").read()[-500:])
    total = sum(mp4_stats(RAW / FOLDER[d] / c / "videos")[1] for d, c in JOBS)
    rate = (total - gb_start) * 1000 / max(time.time() - t_start, 1)
    log(f"   total so far {total:.2f} GB | speed {rate:.1f} MB/s | parts done {sum(status.values())}/{len(JOBS)}")
log(f"All parts processed. OK: {sum(status.values())}/{len(JOBS)} (re-run this cell to retry any CHECK parts)")

In [ ]:
# ===== CELL 5: Count check against the official numbers =====
rows = []
for d, c in JOBS:
    n, gb = mp4_stats(RAW / FOLDER[d] / c / "videos")
    rows.append(dict(dataset=d, compression=c, files=n, expected=EXPECTED[d], GB=round(gb, 2),
                     check="OK" if n == EXPECTED[d] else "MISMATCH"))
counts = pd.DataFrame(rows)
print(counts.to_string(index=False))
print(f"\nTotal: {counts.files.sum():,} files, {counts.GB.sum():.1f} GB")
counts.to_csv(RAW / "_counts.csv", index=False)

In [ ]:
# ===== CELL 6: SHA-256 manifest of the official copy (resumable) =====
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

man_csv = RAW / "MANIFEST.csv"
files = sorted(str(f) for f in RAW.rglob("*.mp4"))
done = pd.read_csv(man_csv) if man_csv.exists() else pd.DataFrame(columns=["rel", "bytes", "sha256"])
have = set(done.rel)
todo = [f for f in files if str(Path(f).relative_to(RAW)) not in have]
log(f"Files: {len(files):,}; already hashed: {len(have):,}; to hash: {len(todo):,}")
buf = []
with ThreadPoolExecutor(8) as ex:
    futs = {ex.submit(sha256, f): f for f in todo}
    for k, fu in enumerate(as_completed(futs), 1):
        f = futs[fu]
        buf.append(dict(rel=str(Path(f).relative_to(RAW)), bytes=os.path.getsize(f), sha256=fu.result()))
        if k % 1000 == 0 or k == len(todo):
            done = pd.concat([done, pd.DataFrame(buf)], ignore_index=True); buf = []
            done.to_csv(man_csv, index=False)
            log(f"  hashed {k:,}/{len(todo):,}")
official = pd.read_csv(man_csv)
log(f"MANIFEST.csv: {len(official):,} files, {official.bytes.sum()/1e9:.1f} GB → {man_csv}")

In [ ]:
# ===== CELL 7: Compare with the Kaggle copy, byte for byte =====
# Needs the audit results: download ffpp_audit_results.zip from the Kaggle notebook's Output tab
# and put it in My Drive/DeepGuard/audit/. Files are matched by method + compression + video name.
METHODS = ["DeepFakeDetection", "Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
def key_of(rel):
    """Map any path layout to the same key, e.g. 'Deepfakes|c23|071_054'."""
    low, stem = rel.lower(), Path(rel).stem
    comp = (re.findall(r"(?:^|[^a-z0-9])(c23|c40|raw|c0)(?:$|[^a-z0-9])", low) or ["?"])[-1]
    if re.fullmatch(r"\d{2}_\d{2}__.+__[A-Za-z0-9]{8}", stem):
        meth = "DeepFakeDetection"
    elif re.fullmatch(r"\d{2}__.+", stem):
        meth = "DFD_original"
    elif re.fullmatch(r"\d{3}", stem):
        meth = "original"
    else:
        meth = next((m for m in METHODS if m.lower() in low), "?")
    return f"{meth}|{comp}|{stem}"

zips = sorted(AUDIT.glob("*.zip"))
if not zips:
    print("No audit zip in", AUDIT, "— put ffpp_audit_results.zip there and re-run this cell.")
else:
    with zipfile.ZipFile(zips[-1]) as z:
        name = next(n for n in z.namelist() if n.endswith("MANIFEST.csv"))
        kag = pd.read_csv(z.open(name))
    official["key"] = official.rel.map(key_of); kag["key"] = kag.rel.map(key_of)
    cmp = official.merge(kag, on="key", how="outer", suffixes=("_official", "_kaggle"), indicator=True)
    cmp["result"] = cmp._merge.map({"left_only": "only in official", "right_only": "only in Kaggle copy",
                                    "both": "both"}).astype(str)
    both = cmp._merge == "both"
    cmp.loc[both, "result"] = (cmp.sha256_official[both] == cmp.sha256_kaggle[both]).map(
        {True: "IDENTICAL", False: "DIFFERENT bytes"})
    cmp["group"] = cmp.key.str.rsplit("|", n=1).str[0]
    print(pd.crosstab(cmp.group, cmp.result).to_string())
    cmp.drop(columns="_merge").to_csv(AUDIT / "official_vs_kaggle.csv", index=False)
    n_same = (cmp.result == "IDENTICAL").sum()
    log(f"Identical: {n_same:,} of {len(official):,} official files → details in {AUDIT/'official_vs_kaggle.csv'}")
    if n_same == len(official) == len(kag):
        print("\nVERDICT: the Kaggle copy is byte-identical to the official download → keep using it on Kaggle.")
    else:
        print("\nVERDICT: not identical — send Claude the table above; the official copy will be uploaded to Kaggle instead.")